In [ ]:
import os
import numpy as np
import pandas as pd

###############################################
input_folder = "" #input path for DLC csv
protocol_path = ""  #input path for protocol csv eg trialLoops.csv
output_folder = os.path.join(input_folder, 'results') # Save results in a subfolder
output_prefix = "R"
###############################################

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def loaddata_dlc(url):
    data = np.delete(np.loadtxt(open(url, 'rb'), delimiter=',', skiprows=3), 0, axis=1)
    LE_angle = []
    RE_angle = []
    for i in range(len(data)):
        if data[i][1] != data[i][4]:  # Avoid division by zero for LE
            LE_angle.append(360 * (np.arctan((data[i][0] - data[i][3]) / (data[i][1] - data[i][4]))) / (2 * np.pi))
        else:
            LE_angle.append(np.nan)  # Assign NaN if division by zero

        if data[i][13] != data[i][16]:  # Avoid division by zero for RE
            RE_angle.append(360 * (np.arctan((data[i][12] - data[i][15]) / (data[i][13] - data[i][16]))) / (2 * np.pi))
        else:
            RE_angle.append(np.nan)  # Assign NaN if division by zero
    return LE_angle, RE_angle

def protocol_load(path):
    pro = pd.read_csv(path)
    duration = pro['gratingDuration']
    orientation = pro['gratingOrientation']
    frequency = pro['gratingSpatialFreq']
    return duration, orientation, frequency

def make_same_length(time, LEdlc, REdlc, duration, orientation, frequency):
    dur = []
    ori = []
    freq = []
    for i in range(len(duration)):
        length = int(duration[i] * 30)
        dur.extend([duration[i]] * length)
        ori.extend([orientation[i]] * length)
        freq.extend([frequency[i]] * length)
    
    min_len = min(len(time), len(dur), len(LEdlc), len(REdlc))
    return time[:min_len], LEdlc[:min_len], REdlc[:min_len], dur[:min_len], ori[:min_len], freq[:min_len]

# Load protocol data once
duration, orientation, frequency = protocol_load(protocol_path)

# Process each CSV file in the input folder
for file_name in os.listdir(input_folder):
    if file_name.endswith('.csv'):
        input_file_path = os.path.join(input_folder, file_name)
        
        # Load DLC data
        LEdlc, REdlc = loaddata_dlc(input_file_path)
        
        # Generate the time array based on the length of the LE angles divided by the frame rate
        time = np.arange(0, len(LEdlc) / 30, 1 / 30)
        
        # Ensure all arrays have the same length
        time, LEdlc, REdlc, dur, ori, freq = make_same_length(time, LEdlc, REdlc, duration, orientation, frequency)
        
        # Create the final DataFrame
        df = pd.DataFrame()
        df['Time'] = time
        df['LE_deg'] = LEdlc
        df['RE_deg'] = REdlc
        df['duration'] = dur
        df['orientation'] = ori
        df['frequency'] = freq
        
        # Save the DataFrame to a new CSV file with the output prefix
        output_file_path = os.path.join(output_folder, output_prefix + file_name)
        df.to_csv(output_file_path, index=False)


In [ ]:
import os
import pandas as pd

# Define the directory containing the CSV files
directory = '"input path.....\\results'" #change the file path here

# Initialize empty lists to store data
time = []
duration = []
orientation = []
frequency = []
LE_data = []
RE_data = []
LE_columns = []
RE_columns = []

# Read each CSV file and process the data
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath)
        
        # Extract and store common data
        if not time:  # only if time list is empty
            time = df.iloc[:, 0].tolist()
            duration = df.iloc[:, 3].tolist()
            orientation = df.iloc[:, 4].tolist()
            frequency = df.iloc[:, 5].tolist()
        
        # Extract and store LE and RE data with column names
        LE_data.append(df.iloc[:, 1].tolist())
        RE_data.append(df.iloc[:, 2].tolist())
        LE_columns.append(f'LE_{filename}')
        RE_columns.append(f'RE_{filename}')

# Combine all LE and RE data into respective dataframes
LE_df = pd.DataFrame(LE_data).T
LE_df.columns = LE_columns
RE_df = pd.DataFrame(RE_data).T
RE_df.columns = RE_columns

# Combine all data into a master dataframe
master_df = pd.DataFrame({
    'time': time,
    'duration': duration,
    'orientation': orientation,
    'frequency': frequency
})

# Concatenate LE and RE dataframes to master dataframe
master_df = pd.concat([master_df, LE_df, RE_df], axis=1)

# Save the master dataframe to a CSV file
master_csv_path = os.path.join(directory, 'master_results.csv')
master_df.to_csv(master_csv_path, index=False)

print(f'Master CSV created at: {master_csv_path}')
